In [1]:
import pandas as pd

df = pd.read_json("alerts.json", lines=True)

# Sysmon events contain nested Windows fields under data.win.*
def is_sysmon(row):
    d = row.get("data", {})
    if not isinstance(d, dict):
        return False
    win = d.get("win", {})
    return isinstance(win, dict) and "system" in win and "eventdata" in win

df_sysmon = df[df.apply(is_sysmon, axis=1)].copy()

print("Total alerts:", len(df))
print("Sysmon alerts:", len(df_sysmon))


Total alerts: 8585
Sysmon alerts: 7838


In [2]:
attack_windows = [
    {"host": "SocWinEP", "start": "2025-12-02 18:03:00", "end": "2025-12-02 18:06:00"},
    {"host": "SocWinEP", "start": "2025-12-02 18:11:00", "end": "2025-12-02 18:12:00"},
    {"host": "SocWinEP", "start": "2025-12-02 18:28:00", "end": "2025-12-02 18:29:00"},
    {"host": "SocWinEP", "start": "2025-12-02 18:26:00", "end": "2025-12-02 18:28:00"},
    {"host": "SocWinEP", "start": "2025-12-02 18:42:00", "end": "2025-12-02 18:46:00"}
    
]

In [3]:
df = df_sysmon.copy()

# timestamp
df["ts"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")

# host name from agent.name
df["computer"] = df["agent"].apply(
    lambda a: a.get("name") if isinstance(a, dict) else None
)

In [4]:
df["label"] = 0  # 0 = benign

In [5]:
delta = pd.Timedelta(seconds=20)

for w in attack_windows:
    host = w["host"]
    start = pd.to_datetime(w["start"], utc=True) - delta
    end   = pd.to_datetime(w["end"],   utc=True) + delta

    mask = (df["computer"] == host) & df["ts"].between(start, end)
    df.loc[mask, "label"] = 1   # 1 = malicious


In [6]:
df['label'].value_counts()

label
0    7101
1     737
Name: count, dtype: int64

In [7]:
df.columns

Index(['timestamp', 'rule', 'agent', 'manager', 'id', 'decoder', 'data',
       'location', 'full_log', 'syscheck', 'previous_output', 'predecoder',
       'ts', 'computer', 'label'],
      dtype='object')

In [8]:
mal = df[df["label"] == 1]

# Inspect a few command lines / images
mal_cmds = mal["data"].apply(lambda d: d["win"]["system"].get("message", "")).sample(600)
malist=mal_cmds.to_list()
print(malist[250])


"File created:
RuleName: -
UtcTime: 2025-12-02 18:02:57.524
ProcessGuid: {8206e0bc-29d1-692f-cf0c-000000000f00}
ProcessId: 11840
Image: C:\Windows\System32\WindowsPowerShell\v1.0\powershell.exe
TargetFilename: C:\Users\Administrator\AppData\Local\Temp\__PSScriptPolicyTest_o4jo4gdz.yz0.ps1
CreationUtcTime: 2025-12-02 18:02:57.524
User: SOC\Administrator"


In [ ]:
files = ["cmd_service_mod_fax_2020-10-2120454410.json", "empire_invoke_runas_2019-05-18204300.json", "empire_uac_shellapi_fodhelper_2020-09-04032946.json"]  # replace with real paths
m_list = [pd.read_json(f, lines=True) for f in files]
mordor_raw = pd.concat(m_list, ignore_index=True)

def is_mordor_sysmon(row):
    chan = str(row.get("Channel", "")).lower()
    return "sysmon" in chan   # detects Microsoft-Windows-Sysmon/Operational


mordor_sysmon = mordor_raw[mordor_raw.apply(is_mordor_sysmon, axis=1)].copy()
print("Sysmon alerts:", len(mordor_sysmon))
print("Total alerts:", len(mordor_raw))



Sysmon alerts: 3259
Total alerts: 7157


In [10]:
mordor_raw.columns

Index(['@timestamp', '@metadata', 'task', 'level', 'keywords', 'beat', 'host',
       'event_id', 'record_number', 'message',
       ...
       'EventCountTotal', 'SessionId', 'ShareLocalPath', 'ShareName',
       'RelativeTargetName', 'TimeCreated', 'Level', 'ProcessID', 'Session',
       'ClientInfo'],
      dtype='object', length=201)

In [ ]:
def map_wazuh_sysmon(row):
    d = row.get("data", {}).get("win", {})
    system = d.get("system", {})
    ev     = d.get("eventdata", {})
    rule      =row.get("rule", {})

    return {
        "ts": pd.to_datetime(row.get("timestamp"), utc=True, errors="coerce"),
        "computer": row.get("computer"),
        "event_id": system.get("eventID"),
        "image": ev.get("image"),
        "command_line": ev.get("commandLine"),
        "target_object": ev.get("targetObject"),
        "message": system.get("message", ""),
        "level": rule.get("level"),
        "source": "wazuh",
        "label": row.get("label", 0),
    }

wazuh_u = df.apply(map_wazuh_sysmon, axis=1, result_type="expand")

In [12]:
n_wazuh_mal = (wazuh_u["label"] == 1).sum()
print(n_wazuh_mal)

737


In [13]:
def map_mordor_sysmon(row):
    return {
        "ts": pd.to_datetime(row.get("UtcTime") or row.get("@timestamp"), utc=True, errors="coerce"),
        "computer": row.get("Hostname"),
        "event_id": row.get("EventID"),
        "image": row.get("Image"),
        "command_line": row.get("CommandLine"),
        "target_object": row.get("TargetObject"),
        "message": row.get("Message", ""),
        'level': row.get('Level'),
        "source": "mordor",
        "label": 1,   # all Mordor here are malicious
    }

mordor_u = mordor_sysmon.apply(map_mordor_sysmon, axis=1, result_type="expand")


In [14]:
mordor_sample = mordor_u.sample(n=n_wazuh_mal, random_state=42)

In [15]:
len(mordor_sample)

737

In [16]:
combined = pd.concat([wazuh_u, mordor_sample], ignore_index=True)

In [17]:
print(combined["label"].value_counts())

label
0    7101
1    1474
Name: count, dtype: int64


In [18]:
print(combined["source"].value_counts())

source
wazuh     7838
mordor     737
Name: count, dtype: int64


In [19]:
print(mordor_sample["message"])

4832    Registry object added or deleted:\r\nRuleName:...
2800    Process accessed:\r\nRuleName: -\r\nUtcTime: 2...
3758    Image loaded:\r\nRuleName: -\r\nUtcTime: 2020-...
5744    Process accessed:\r\nRuleName: -\r\nUtcTime: 2...
6925    Registry object added or deleted:\r\nRuleName:...
                              ...                        
6536    Process accessed:\r\nRuleName: -\r\nUtcTime: 2...
3105    Image loaded:\r\nRuleName: -\r\nUtcTime: 2020-...
4790    Registry object added or deleted:\r\nRuleName:...
6829    Process accessed:\r\nRuleName: -\r\nUtcTime: 2...
5425    Process accessed:\r\nRuleName: -\r\nUtcTime: 2...
Name: message, Length: 737, dtype: object


In [20]:
combined.columns

Index(['ts', 'computer', 'event_id', 'image', 'command_line', 'target_object',
       'message', 'level', 'source', 'label'],
      dtype='object')

In [21]:
mordor_sample["image"].value_counts()

image
C:\Windows\System32\WindowsPowerShell\v1.0\powershell.exe                                                                       107
C:\windows\System32\WindowsPowerShell\v1.0\powershell.exe                                                                        70
C:\windows\system32\WindowsPowerShell\v1.0\powershell.exe                                                                        40
C:\windows\system32\consent.exe                                                                                                  37
C:\windows\system32\svchost.exe                                                                                                  29
System                                                                                                                           26
C:\Windows\System32\whoami.exe                                                                                                   24
C:\Windows\System32\svchost.exe                                       

In [22]:
mordor_sample["event_id"].value_counts

<bound method IndexOpsMixin.value_counts of 4832    12.0
2800    10.0
3758     7.0
5744    10.0
6925    12.0
        ... 
6536    10.0
3105     7.0
4790    12.0
6829    10.0
5425    10.0
Name: event_id, Length: 737, dtype: float64>

In [23]:
wazuh_u.to_csv("wazuh_only.csv", index=False)

In [24]:
combined.to_csv("wazuh_mordor_comb.csv", index=False)

In [25]:
missing = (mordor_sample.isna().mean() * 100).sort_values(ascending=False)
print(missing.head(15))

command_line     99.185889
level            88.873813
target_object    65.264586
image            31.343284
computer          0.000000
ts                0.000000
event_id          0.000000
message           0.000000
source            0.000000
label             0.000000
dtype: float64


In [26]:
missing = (wazuh_u.isna().mean() * 100).sort_values(ascending=False)
print(missing.head(15))

target_object    87.241643
command_line     81.704516
image            62.413881
ts                0.000000
computer          0.000000
event_id          0.000000
message           0.000000
level             0.000000
source            0.000000
label             0.000000
dtype: float64
